# Python Bridge for 06b–07a

## A Short Code Refresher

Use this optional notebook alongside 06b and 07a for a closer look at the Python mechanics of K-means. Plan for about 15–20 minutes. All eight made-up customer records are supplied below.

The lectures explain what a K-means partition means and how to assess it. This companion follows five code patterns:

1. select and scale the input features;
2. create and fit a K-means object;
3. attach labels and inspect centers;
4. distinguish distances from assignments; and
5. assign a new observation using the fitted scaler and model.

## References

The scikit-learn references for [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) and [KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html) describe the methods and stored attributes used here. The [Course Addenda and Errata page](https://auburn.instructure.com/courses/1747056/pages/course-addenda-and-errata) provides the corresponding 07A clarification of fitting, transforming, and predicting.

## Create a Feature Table

Each row describes one customer's support requests during the same period. The two features are request count and average minutes per request. The customer ID identifies a row; it is not an input feature.

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [ ]:
customers = pd.DataFrame(
    {
        "requests": [2, 3, 4, 3, 16, 18, 20, 18],
        "avg_minutes": [4, 6, 5, 5, 14, 16, 15, 15],
    },
    index=list("ABCDEFGH"),
)
feature_columns = ["requests", "avg_minutes"]
features = customers[feature_columns]

print("feature-table shape:", features.shape)
features

The input shape is `(8, 2)`: eight observations, each described by two features. These deliberately separated records make the returned objects easy to inspect. They do not reproduce the independently generated customer sample in the lectures.

## Fit and Apply the Scaler

`StandardScaler()` creates an object that will store one mean and scale per feature. `.fit(features)` learns those values from these eight rows. `.transform(features)` applies them and returns a numerical array in the same row and column order.

In [ ]:
scaler = StandardScaler()
scaler.fit(features)
scaled = scaler.transform(features)

print("scaled-array shape:", scaled.shape)
pd.DataFrame({"Mean": scaler.mean_, "Scale": scaler.scale_}, index=feature_columns).round(3)

The trailing underscores identify learned attributes. `StandardScaler` uses the population standard deviation (`ddof=0`), whereas the 04a bridge used pandas sample standard deviations (`ddof=1`). Here we follow the clustering lectures' scaler. Keep it with the fitted clustering model so future rows receive the same preprocessing.

In [ ]:
pd.DataFrame(scaled, index=features.index, columns=feature_columns).round(3)

Negative standardized values lie below the corresponding fitted mean. The array has no customer labels of its own, so the display restores them from `features.index`.

## Create and Fit the K-Means Object

`n_clusters=2` requests two centers. `n_init=20` requests twenty starts and retains the result with the smallest inertia. `random_state=7130` makes the random choices repeatable. These are supplied settings; fitting learns the centers and assignments.

In [ ]:
model = KMeans(n_clusters=2, n_init=20, random_state=7130)
model.fit(scaled)

print("center-array shape:", model.cluster_centers_.shape)
print("label-array shape:", model.labels_.shape)
print("inertia:", round(model.inertia_, 3))

There are two centers with two coordinates each and one label per customer. `.fit()` returns the fitted object, not a label array. Inertia is the sum of squared distances to assigned centers in the standardized feature space.

## Attach Labels to the Original Rows

`model.labels_` follows the row order used during fitting. `.assign()` makes a table with those labels attached. The labels identify groups; their numerical order does not rank the customers.

In [ ]:
labeled = customers.assign(cluster=model.labels_)

labeled

A–D share one label, and E–H share the other. Read those memberships rather than relying on which group receives label 0.

In [ ]:
profiles = labeled.groupby("cluster")[feature_columns].mean()
centers_original = pd.DataFrame(
    scaler.inverse_transform(model.cluster_centers_), columns=feature_columns
).rename_axis("cluster")

print("Means of the assigned customer rows:")
print(profiles.round(3))
print("Fitted centers restored to original units:")
print(centers_original.round(3))

The two displays agree for this converged fit. `inverse_transform` reverses the scaling, making the centers readable as requests and minutes. The low-request group averages 3 requests and 5 minutes; the other averages 18 requests and 15 minutes.

## Distinguish Distances From Assignments

A K-means model's `.transform()` returns a distance to every center. Its `.predict()` returns the label of the nearest center. Both use the centers already stored on the model.

In [ ]:
distances = model.transform(scaled)
assignments = model.predict(scaled)
distance_columns = [f"Distance to center {i}" for i in range(model.n_clusters)]
distance_table = pd.DataFrame(distances, index=features.index, columns=distance_columns)
distance_table["Predicted label"] = assignments

print("distance shape:", distances.shape)
print("assignment shape:", assignments.shape)
distance_table.round(3)

The distances have shape `(8, 2)`, while the assignments have shape `(8,)`. Each label selects that row's smallest distance. The distances are ordinary Euclidean distances; inertia squares the selected distances and sums them. A scaler's `transform` and a K-means model's `transform` therefore perform different operations.

## Assign a New Customer

Supply the same feature columns in the same order. First use the existing scaler, then ask the existing K-means model for an assignment. Neither object is fitted again.

In [ ]:
new_customers = pd.DataFrame({"requests": [5], "avg_minutes": [6]}, index=["New"])
new_scaled = scaler.transform(new_customers[feature_columns])
new_labels = model.predict(new_scaled)

print("new-input shape:", new_scaled.shape)
print("distances to fitted centers:", model.transform(new_scaled).round(3))
new_customers.assign(cluster=new_labels)

This customer joins the group containing A–D. Fitting a scaler on the new row would give it a different coordinate system. Fitting K-means again would estimate a new partition. Prediction assigns the row to the existing partition, although usefulness still requires checking whether the row is reasonably represented by those groups.

## Try One Small Modification

Change the new customer's values to 19 requests and 14 minutes, then rerun only the new-customer cell. It should join E–H. The stored centers, scaling values, and original customer assignments should remain unchanged.

## Ready for 06b–07a

You are ready to return to the lectures when you can recognize these patterns:

```text
scaler.fit(features)                  # learn feature means and scales
scaled = scaler.transform(features)  # apply the saved preprocessing
model.fit(scaled)                     # learn centers and assignments
model.labels_                        # inspect fitted-row assignments
model.transform(scaled)              # return distances to every center
model.predict(new_scaled)            # assign rows using the fitted centers
```

The output's meaning depends on which object received the call. Preserve the feature order, fitted scaler, and fitted clustering model together.

---

Auburn University / Industrial and Systems Engineering<br>
INSY 7130, Pattern Discovery and Time Series Analysis<br>
© Copyright Danny J. O'Leary.

For course materials, attribution, and licensing information, see the [INSY 7130 course-materials README](../../README.md).